# 5-2. ResNet 계열 확장 이해하기

이 노트북은 `05_ResNet_Deep_CNN.ipynb` 다음 단계로, **ResNet이 어떤 방향으로 확장되었는지**를 이어서 살펴보는 실습입니다.

이전 노트북에서는 `skip connection`이 왜 깊은 CNN 학습을 쉽게 만드는지 확인했습니다. 이번에는 그 다음 질문인 **"그렇다면 ResNet은 어떤 식으로 더 깊어지고, 더 넓어지고, 더 다양한 방향으로 확장되었을까?"** 에 답해 보겠습니다.

이번 노트북의 목표는 다음과 같습니다.

- `BasicBlock`과 `Bottleneck block`의 차이를 이해합니다.
- `ResNet-18`, `ResNet-50`, `Wide ResNet`, `ResNeXt`를 비교합니다.
- `torchvision.models`의 표준 ResNet 계열 모델을 CIFAR-10 분류 문제에 맞게 바꾸어 봅니다.
- 필요하면 작은 실험으로 어떤 계열을 먼저 써 볼지 감을 잡습니다.


## 5-2-1. ResNet은 어떻게 확장될까?

ResNet 계열은 크게 네 가지 방향으로 확장된다고 볼 수 있습니다.

- **더 깊게**: `ResNet-18`, `34`, `50`, `101`, `152`처럼 block 수를 늘립니다.
- **더 효율적으로**: `Bottleneck block`을 사용해 깊이는 유지하면서 계산 구조를 정리합니다.
- **더 넓게**: `Wide ResNet`처럼 각 block 안 채널 수를 늘려 표현력을 키웁니다.
- **더 다양한 경로로**: `ResNeXt`처럼 group convolution을 사용해 병렬 경로 수(cardinality)를 늘립니다.

즉, ResNet 계열은 단순히 층 수만 늘린 모델이 아니라, **깊이(depth)**, **너비(width)**, **경로 수(cardinality)** 라는 세 축으로 발전한 구조라고 이해하면 좋습니다.


In [ ]:
# 필요 라이브러리가 없다면 아래 주석을 해제해서 설치하세요.
# !pip install torch torchvision matplotlib


In [ ]:
import copy
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, Subset


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('사용 장치:', device)


## 5-2-2. 데이터 준비

이번에는 `torchvision.models`의 표준 ResNet 계열을 그대로 써 보기 위해 입력 이미지를 `224 x 224`로 키웁니다. 원래 CIFAR-10은 `32 x 32`이지만, 표준 ImageNet용 ResNet 구조를 연습하는 데에는 이 방식이 가장 직관적입니다.

학습 시간 부담을 줄이기 위해 기본 설정은 **작은 subset** 으로 두었습니다. 더 진지하게 실험하고 싶다면 `train_size`, `val_size`, `selected_model_names`, `epochs`를 늘리면 됩니다.


In [ ]:
image_size = 224
train_size = 4000
val_size = 1000
batch_size = 32

mean = (0.4914, 0.4822, 0.4465)
std = (0.2470, 0.2435, 0.2616)

train_transform = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

eval_transform = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

full_train_aug = datasets.CIFAR10(root='./data', train=True, download=True, transform=train_transform)
full_train_eval = datasets.CIFAR10(root='./data', train=True, download=False, transform=eval_transform)
test_dataset = datasets.CIFAR10(root='./data', train=False, download=True, transform=eval_transform)

classes = full_train_aug.classes

generator = torch.Generator().manual_seed(42)
all_indices = torch.randperm(len(full_train_aug), generator=generator).tolist()
train_indices = all_indices[:train_size]
val_indices = all_indices[train_size:train_size + val_size]

train_dataset = Subset(full_train_aug, train_indices)
val_dataset = Subset(full_train_eval, val_indices)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size * 2, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size * 2, shuffle=False)

print('Classes:', classes)
print('Train samples:', len(train_dataset))
print('Validation samples:', len(val_dataset))
print('Test samples:', len(test_dataset))


In [ ]:
def denormalize(image):
    mean_tensor = torch.tensor(mean).view(3, 1, 1)
    std_tensor = torch.tensor(std).view(3, 1, 1)
    return (image.cpu() * std_tensor + mean_tensor).clamp(0, 1)

images, labels = next(iter(train_loader))

fig, axes = plt.subplots(2, 4, figsize=(10, 5))
for ax, image, label in zip(axes.flat, images[:8], labels[:8]):
    ax.imshow(denormalize(image).permute(1, 2, 0))
    ax.set_title(classes[label])
    ax.axis('off')
plt.tight_layout()
plt.show()


## 5-2-3. BasicBlock과 Bottleneck block 비교

`ResNet-18`, `ResNet-34`는 주로 `BasicBlock`을 사용합니다. 반면 `ResNet-50` 이상에서는 `Bottleneck block`이 많이 쓰입니다.

`Bottleneck`의 핵심은 `1x1 -> 3x3 -> 1x1` 구조입니다.

- 첫 번째 `1x1 convolution`: 채널 수를 줄여 계산량을 압축합니다.
- 가운데 `3x3 convolution`: 실제 공간적 특징을 학습합니다.
- 마지막 `1x1 convolution`: 채널 수를 다시 늘려 다음 stage로 넘깁니다.

즉, **깊이는 늘리되 계산량을 통제하기 위한 구조적 장치** 가 bottleneck입니다.


In [ ]:
class BasicResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)

        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )
        else:
            self.shortcut = nn.Identity()

    def forward(self, x):
        identity = self.shortcut(x)
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out = self.relu(out + identity)
        return out


class BottleneckResidualBlock(nn.Module):
    expansion = 4

    def __init__(self, in_channels, bottleneck_channels, stride=1):
        super().__init__()
        out_channels = bottleneck_channels * self.expansion

        self.conv1 = nn.Conv2d(in_channels, bottleneck_channels, kernel_size=1, bias=False)
        self.bn1 = nn.BatchNorm2d(bottleneck_channels)
        self.conv2 = nn.Conv2d(bottleneck_channels, bottleneck_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(bottleneck_channels)
        self.conv3 = nn.Conv2d(bottleneck_channels, out_channels, kernel_size=1, bias=False)
        self.bn3 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)

        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )
        else:
            self.shortcut = nn.Identity()

    def forward(self, x):
        identity = self.shortcut(x)
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.relu(self.bn2(self.conv2(out)))
        out = self.bn3(self.conv3(out))
        out = self.relu(out + identity)
        return out


def count_parameters(model):
    return sum(param.numel() for param in model.parameters() if param.requires_grad)


basic_block = BasicResidualBlock(64, 64)
bottleneck_block = BottleneckResidualBlock(64, 64)
sample = torch.randn(2, 64, 56, 56)

with torch.no_grad():
    basic_out = basic_block(sample)
    bottleneck_out = bottleneck_block(sample)

print('BasicBlock 출력 shape :', tuple(basic_out.shape))
print('Bottleneck 출력 shape:', tuple(bottleneck_out.shape))
print('BasicBlock 파라미터 수 :', f'{count_parameters(basic_block):,}')
print('Bottleneck 파라미터 수:', f'{count_parameters(bottleneck_block):,}')


위 결과에서 눈여겨볼 점은 `Bottleneck`이 **중간 채널을 줄였다가 다시 키우는 구조** 라는 것입니다. 실제 대형 ResNet에서는 이 패턴 덕분에 깊이를 늘리면서도 계산량을 상대적으로 효율적으로 관리할 수 있습니다.


## 5-2-4. 표준 ResNet 계열 모델 준비

이제 `torchvision.models`에서 제공하는 대표적인 ResNet 계열 모델을 가져옵니다.

- `resnet18`: basic block 기반의 비교적 가벼운 모델
- `resnet50`: bottleneck 기반의 대표 모델
- `wide_resnet50_2`: block 내부 채널 폭을 넓힌 모델
- `resnext50_32x4d`: group convolution으로 cardinality를 늘린 모델

여기서는 ImageNet용 마지막 분류기(`fc`)만 CIFAR-10용으로 바꾸어 사용합니다.


In [ ]:
def build_resnet18(num_classes=10):
    model = models.resnet18(weights=None)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model


def build_resnet50(num_classes=10):
    model = models.resnet50(weights=None)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model


def build_wide_resnet50_2(num_classes=10):
    model = models.wide_resnet50_2(weights=None)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model


def build_resnext50_32x4d(num_classes=10):
    model = models.resnext50_32x4d(weights=None)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model


model_builders = {
    'resnet18': build_resnet18,
    'resnet50': build_resnet50,
    'wide_resnet50_2': build_wide_resnet50_2,
    'resnext50_32x4d': build_resnext50_32x4d,
}

model_summaries = []
for name, builder in model_builders.items():
    model = builder(num_classes=len(classes))
    summary = {
        'name': name,
        'block': type(model.layer1[0]).__name__,
        'params_m': count_parameters(model) / 1_000_000,
        'fc_in_features': model.fc.in_features,
        'groups': getattr(model, 'groups', 1),
        'base_width': getattr(model, 'base_width', 64),
    }
    model_summaries.append(summary)

for summary in model_summaries:
    print(
        f"{summary['name']:<18} | block={summary['block']:<10} | "
        f"params={summary['params_m']:.2f}M | fc_in={summary['fc_in_features']:<4} | "
        f"groups={summary['groups']:<2} | base_width={summary['base_width']}"
    )

plt.figure(figsize=(8, 4))
plt.bar(
    [summary['name'] for summary in model_summaries],
    [summary['params_m'] for summary in model_summaries],
    color=['#2563eb', '#0f766e', '#f59e0b', '#7c3aed']
)
plt.ylabel('Parameters (Millions)')
plt.title('ResNet 계열 파라미터 수 비교')
plt.xticks(rotation=15)
plt.show()


In [ ]:
def inspect_feature_shapes(model_name, input_size=224):
    model = model_builders[model_name](num_classes=len(classes)).to(device)
    model.eval()
    x = torch.randn(1, 3, input_size, input_size, device=device)
    shapes = {}

    with torch.no_grad():
        x = model.conv1(x)
        x = model.bn1(x)
        x = model.relu(x)
        shapes['stem'] = tuple(x.shape)

        x = model.maxpool(x)
        shapes['maxpool'] = tuple(x.shape)

        x = model.layer1(x)
        shapes['layer1'] = tuple(x.shape)
        x = model.layer2(x)
        shapes['layer2'] = tuple(x.shape)
        x = model.layer3(x)
        shapes['layer3'] = tuple(x.shape)
        x = model.layer4(x)
        shapes['layer4'] = tuple(x.shape)

        x = model.avgpool(x)
        shapes['avgpool'] = tuple(x.shape)

    return shapes


for model_name in model_builders:
    print(f'=== {model_name} ===')
    shapes = inspect_feature_shapes(model_name, input_size=image_size)
    for stage, shape in shapes.items():
        print(f'{stage:<8}: {shape}')
    print()


출력 shape를 보면 stage별 해상도 감소 패턴은 대체로 비슷합니다. 대신 각 계열은 **block 내부 계산 방식** 이 다릅니다.

- `ResNet-18`: 가장 단순하고 가볍게 시작하기 좋습니다.
- `ResNet-50`: bottleneck 구조로 더 깊고 표현력이 큽니다.
- `Wide ResNet`: 같은 stage라도 block 내부 채널 폭이 더 넓습니다.
- `ResNeXt`: 넓이를 마구 키우기보다 group convolution으로 여러 경로를 병렬화합니다.

실전에서는 데이터 크기, 연산 예산, 추론 속도 요구에 따라 어떤 축을 늘릴지 결정합니다.


## 5-2-5. 선택적 학습 실험

아래 코드는 실제로 몇 개 모델을 학습해 보는 실험입니다. 기본값은 `run_training = False`로 두었습니다. 이유는 표준 ResNet 계열을 `224 x 224` 입력으로 학습하면 CPU 환경에서 꽤 오래 걸릴 수 있기 때문입니다.

추천 순서는 다음과 같습니다.

- 먼저 `resnet18`만 1 epoch 실행해 흐름을 확인합니다.
- 다음으로 `resnet50`을 추가해 bottleneck 기반 모델을 비교합니다.
- 여유가 있으면 `wide_resnet50_2`, `resnext50_32x4d`까지 늘립니다.


In [ ]:
def evaluate(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            total_loss += loss.item() * labels.size(0)
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    return total_loss / total, correct / total


def train_model(model, train_loader, val_loader, epochs=1, lr=0.001):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    best_state = copy.deepcopy(model.state_dict())
    best_val_acc = 0.0
    history = []

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0

        for images, labels in train_loader:
            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * labels.size(0)
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

        train_loss = running_loss / total
        train_acc = correct / total
        val_loss, val_acc = evaluate(model, val_loader, criterion)
        history.append((train_loss, train_acc, val_loss, val_acc))

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = copy.deepcopy(model.state_dict())

        print(
            f"Epoch {epoch + 1}/{epochs} | "
            f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f} | "
            f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}"
        )

    model.load_state_dict(best_state)
    return history, criterion


In [ ]:
run_training = False
selected_model_names = ['resnet18', 'resnet50']
epochs = 1
learning_rate = 0.001

trained_models = {}
results = {}

if run_training:
    for model_name in selected_model_names:
        print(f'=== {model_name} 학습 ===')
        model = model_builders[model_name](num_classes=len(classes)).to(device)
        history, criterion = train_model(model, train_loader, val_loader, epochs=epochs, lr=learning_rate)
        test_loss, test_acc = evaluate(model, test_loader, criterion)
        trained_models[model_name] = model
        results[model_name] = {
            'history': history,
            'test_loss': test_loss,
            'test_acc': test_acc,
        }
        print(f'{model_name} | Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.4f}')
        print()
else:
    print('run_training=False 이므로 구조 비교까지만 실행합니다. 필요하면 True로 바꿔 학습 실험을 진행하세요.')


In [ ]:
if results:
    model_names = list(results.keys())
    test_accs = [results[name]['test_acc'] for name in model_names]

    plt.figure(figsize=(7, 4))
    plt.bar(model_names, test_accs, color=['#2563eb', '#0f766e', '#f59e0b', '#7c3aed'][:len(model_names)])
    plt.ylim(0, 1)
    plt.ylabel('Accuracy')
    plt.title('선택한 ResNet 계열 Test Accuracy 비교')
    plt.show()

    for model_name in model_names:
        history = results[model_name]['history']
        epochs_axis = range(1, len(history) + 1)
        train_accs = [item[1] for item in history]
        val_accs = [item[3] for item in history]
        plt.plot(epochs_axis, train_accs, marker='o', label=f'{model_name} Train')
        plt.plot(epochs_axis, val_accs, marker='s', linestyle='--', label=f'{model_name} Val')

    plt.title('Epoch별 Accuracy 추이')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.show()
else:
    print('아직 학습 결과가 없습니다. 위 셀에서 run_training을 True로 바꾸면 이 그래프가 활성화됩니다.')


## 5-2-6. 예측 결과 확인

학습을 실행했다면 가장 성능이 좋은 모델의 예측 결과를 직접 확인해 봅니다. 구조 차이가 결국 실제 분류 결과에 어떻게 나타나는지 보는 단계입니다.


In [ ]:
if results:
    best_model_name = max(results, key=lambda name: results[name]['test_acc'])
    best_model = trained_models[best_model_name]
    best_model.eval()

    images, labels = next(iter(test_loader))
    images = images.to(device)
    labels = labels.to(device)

    with torch.no_grad():
        outputs = best_model(images)
        preds = outputs.argmax(dim=1)

    fig, axes = plt.subplots(2, 4, figsize=(10, 5))
    for ax, image, label, pred in zip(axes.flat, images[:8], labels[:8], preds[:8]):
        ax.imshow(denormalize(image).permute(1, 2, 0))
        ax.set_title(f'T: {classes[label]}\nP: {classes[pred]}')
        ax.axis('off')
    plt.suptitle(f'Best Model: {best_model_name}', y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print('학습을 실행하지 않았으므로 예측 시각화도 건너뜁니다.')


## 정리

이번 노트북의 핵심은 다음과 같습니다.

- `ResNet-18`은 가볍고 이해하기 쉬운 출발점입니다.
- `ResNet-50`은 `Bottleneck block`을 사용해 더 깊은 구조를 효율적으로 만듭니다.
- `Wide ResNet`은 depth만 늘리는 대신 width를 키워 표현력을 높입니다.
- `ResNeXt`는 group convolution으로 **cardinality** 라는 새로운 축을 활용합니다.
- 결국 ResNet 계열 비교는 단순히 "누가 더 깊은가"가 아니라, **깊이 / 너비 / 병렬 경로 수를 어떻게 배분할 것인가** 의 문제입니다.

다음 단계로는 아래와 같은 확장을 해 볼 수 있습니다.

- `pretrained weights`를 사용한 transfer learning 실험
- `ResNet-18`과 `EfficientNet` 또는 `DenseNet` 비교
- `Grad-CAM`으로 ResNet 계열이 어디를 보고 판단하는지 시각화
